# Track 1 starter: Claim classification baseline
TF-IDF + logistic regression on synthetic claims. **Goal:** show the workflow (split, baseline, error analysis). Real solutions need real, ethically sourced data, multilingual coverage and human fact-checkers in the loop.

> Synthetic templates are easy to memorise, so high scores here are NOT evidence of real-world performance.

In [ ]:
import sys; sys.path.append('../src')
import pandas as pd, numpy as np
from ai4elections import load_csv, load_jsonl
pd.set_option('display.max_columns', 30)

In [ ]:
claims = load_jsonl('misinformation_claims_synthetic.jsonl')
claims['label'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

# Split by language-stratified sample; consider group splits to avoid template leakage.
X_tr, X_te, y_tr, y_te = train_test_split(claims['text'], claims['label'], test_size=0.25, stratify=claims['label'], random_state=0)
clf = make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2), LogisticRegression(max_iter=1000))
clf.fit(X_tr, y_tr)
print(classification_report(y_te, clf.predict(X_te)))

## Your turn
1. Break the model: write 10 claims in different phrasing and see what fails.
2. Evaluate per language (`en` vs `pcm`).
3. Add an **abstain** option: return 'needs human review' when confidence < threshold.
4. Document limits in `docs/05_templates/MODEL_CARD.md`.

**Important design rule:** flag content for human fact-checkers. Do not auto-censor or auto-label real citizens' speech as false.

In [ ]:
# Example: confidence-based abstention
proba = clf.predict_proba(X_te); conf = proba.max(axis=1)
threshold = 0.6
abstained = (conf < threshold).mean()
print(f'Abstains on {abstained:.0%} of items at threshold {threshold}')